# Multimodal emotion recognition on TESS (speech, text, fusion)

1. **Add data:** the dataset *Toronto emotional speech set (TESS)* (`ejlok1/toronto-emotional-speech-set-tess`).
2. **Settings:** Accelerator *GPU T4 x1* (or P100), Internet *on* (downloads WavLM and BERT).
3. **Save Version > Save & Run All (Commit).** About 1.5-2 hours.
4. Download `TESS_results.zip` from the Output tab.

In [ ]:
CODE_B64 = '...'
CODE_BUILD = '2026-11-16 21:37 2391b93e'

In [ ]:
import base64, io, os, sys, tarfile, subprocess, time, torch
PROJ = '/kaggle/working/project'
os.makedirs(PROJ, exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode='r:gz').extractall(PROJ)
print('code build:', CODE_BUILD)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (enable a GPU!)')
import transformers, librosa, sklearn
print('torch', torch.__version__, '| transformers', transformers.__version__, '| librosa',
      librosa.__version__, '| sklearn', sklearn.__version__)
wavs = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs if f.lower().endswith('.wav')]
print(len(wavs), 'wav files under /kaggle/input, e.g.', wavs[:2])
assert wavs, 'TESS not found: add the dataset (Add data -> ejlok1/toronto-emotional-speech-set-tess)'

In [ ]:
def run(cmd):
    print('+', cmd, flush=True)
    t0 = time.time()
    p = subprocess.run(cmd, shell=True, cwd=PROJ)
    print(f'({time.time() - t0:.0f} s)')
    assert p.returncode == 0, f'failed: {cmd}'

FEAT = '/tmp/tess_features'   # about 7 GB, not part of the output
run(f'python scripts/extract_features.py --tess /kaggle/input --out {FEAT}')

In [ ]:
run(f'python scripts/run_all.py --features {FEAT} --results Results --seeds 0 1 2 --epochs 40')

In [ ]:
run(f'python scripts/text_zero_shot.py --features {FEAT} --results Results')

In [ ]:
run('python scripts/make_results.py --results Results')
print(open(f'{PROJ}/Results/accuracy_table.md').read())

In [ ]:
# keep weights of seed 0 for the three main models; every run keeps metrics, predictions, reps, logs
import glob, zipfile
keep = {f'Results/runs/{v}/{s}/seed0/model.pt'
        for v in ('speech', 'speech_spknorm', 'text', 'fusion', 'fusion_xattn')
        for s in ('word', 'oaf2yaf', 'yaf2oaf')}
out = '/kaggle/working/TESS_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f'{PROJ}/Results/**/*', recursive=True):
        rel = os.path.relpath(p, PROJ).replace(os.sep, '/')
        if os.path.isfile(p) and (not rel.endswith('model.pt') or rel in keep):
            z.write(p, rel)
    for f in ('metadata.json', 'splits.json', 'text_tokens.json'):
        z.write(f'{FEAT}/{f}', f'Results/data/{f}')
print(out, round(os.path.getsize(out) / 1e6, 1), 'MB | code build', CODE_BUILD)